# PathWise AI Prototype
## Human-Centred Adaptive Learning for Corporate L&D

This notebook is part of the **Building AI course project**.

It demonstrates a deliberately modest machine-learning workflow using **synthetic learner data**. The purpose is not to build a production recommender, but to show how a probabilistic model can be implemented, evaluated, and translated into a human-centred learning recommendation.

**Important:** Predictions are treated as decision-support signals, not as deterministic judgements of learner ability or employment potential.


## 1. Load the synthetic dataset

The data contain five learner features and one binary outcome:

- diagnostic score
- prior AI experience
- scenario performance
- practice completion
- confidence
- completion of the advanced pathway

The dataset contains no real learner or employee information.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix
)

df = pd.read_csv("../data/synthetic_learners.csv")
df.head()


,learner_id,diagnostic_score,prior_ai_experience,scenario_performance,practice_completion_pct,confidence_1_to_5,completed_advanced_pathway
0,L0001,66.6,0,42.3,71.0,2,0
1,L0002,46.4,1,58.5,40.5,4,0
2,L0003,73.3,0,78.8,71.8,3,0
3,L0004,76.1,0,66.3,27.1,2,1
4,L0005,32.7,2,19.5,47.6,4,1


In [2]:
df["completed_advanced_pathway"].value_counts(normalize=True).rename("proportion")


completed_advanced_pathway
1    0.633333
0    0.366667
Name: proportion, dtype: float64

## 2. Define the prediction task

The prototype asks:

> Given selected learner evidence, what is the estimated probability that the learner completes the advanced pathway?

This is a **prediction problem**, not a causal inference problem. The model does not tell us what intervention would cause better learning outcomes.


In [3]:
features = [
    "diagnostic_score",
    "prior_ai_experience",
    "scenario_performance",
    "practice_completion_pct",
    "confidence_1_to_5",
]
target = "completed_advanced_pathway"

X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))


Training rows: 900
Test rows: 300


## 3. Establish a baseline

A model should outperform a meaningful baseline. Here we use a majority-class classifier that always predicts the most common outcome in the training data.


In [4]:
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)

baseline_pred = baseline.predict(X_test)
baseline_accuracy = accuracy_score(y_test, baseline_pred)

print("Baseline accuracy:", round(baseline_accuracy, 3))


Baseline accuracy: 0.633


## 4. Train logistic regression

Logistic regression is appropriate here because the outcome is binary. The model outputs a probability between 0 and 1.

A pipeline standardises the features before fitting the model.


In [5]:
model = Pipeline([
    ("scale", StandardScaler()),
    ("logistic", LogisticRegression(max_iter=1000, random_state=42))
])

model.fit(X_train, y_train)

pred = model.predict(X_test)
proba = model.predict_proba(X_test)[:, 1]


## 5. Evaluate on unseen test data

Accuracy is not enough by itself. We also inspect precision, recall, F1, and ROC AUC.


In [6]:
results = {
    "baseline_accuracy": baseline_accuracy,
    "model_accuracy": accuracy_score(y_test, pred),
    "precision": precision_score(y_test, pred),
    "recall": recall_score(y_test, pred),
    "f1": f1_score(y_test, pred),
    "roc_auc": roc_auc_score(y_test, proba),
}

pd.Series(results).round(3)


baseline_accuracy    0.633
model_accuracy       0.673
precision            0.704
recall               0.837
f1                   0.764
roc_auc              0.726
dtype: float64

In [7]:
cm = confusion_matrix(y_test, pred)
pd.DataFrame(
    cm,
    index=["Actual 0", "Actual 1"],
    columns=["Predicted 0", "Predicted 1"]
)


,Predicted 0,Predicted 1
Actual 0,43,67
Actual 1,31,159


## 6. Decision thresholds matter

A default threshold of 0.50 is a convention, not a law. In a learning-support context, missing someone who needs support may be more costly than offering unnecessary support.

The next cell compares the default threshold with a lower threshold of 0.35.


In [8]:
def metrics_at_threshold(probabilities, y_true, threshold):
    p = (probabilities >= threshold).astype(int)
    return {
        "threshold": threshold,
        "accuracy": accuracy_score(y_true, p),
        "precision": precision_score(y_true, p),
        "recall": recall_score(y_true, p),
        "f1": f1_score(y_true, p),
    }

comparison = pd.DataFrame([
    metrics_at_threshold(proba, y_test, 0.50),
    metrics_at_threshold(proba, y_test, 0.35),
])

comparison.round(3)


,threshold,accuracy,precision,recall,f1
0,0.50,0.673,0.704,0.837,0.764
1,0.35,0.667,0.663,0.963,0.785


## 7. Inspect coefficients carefully

The coefficients show predictive associations within this fitted model. They should **not** be interpreted as causal effects.

For example, a positive coefficient for practice completion does not prove that forcing every learner to complete more practice will cause a particular increase in completion probability.


In [9]:
coefs = model.named_steps["logistic"].coef_[0]
coef_table = pd.DataFrame({
    "feature": features,
    "standardised_coefficient": coefs
}).sort_values("standardised_coefficient", ascending=False)

coef_table


,feature,standardised_coefficient
0,diagnostic_score,0.538552
3,practice_completion_pct,0.429157
2,scenario_performance,0.311813
1,prior_ai_experience,0.238525
4,confidence_1_to_5,0.193803


## 8. Human-centred recommendation example

The model output should support a recommendation rather than determine a learner's identity or access.

A real interface should explain the recommendation, show uncertainty appropriately, and let the learner review or choose an alternative pathway.


In [10]:
example = X_test.iloc[[0]]
p_complete = model.predict_proba(example)[:, 1][0]

if p_complete < 0.45:
    recommendation = "Foundation pathway recommended"
elif p_complete < 0.70:
    recommendation = "Applied pathway recommended"
else:
    recommendation = "Advanced pathway recommended"

print("Estimated completion probability:", round(p_complete, 3))
print("Recommendation:", recommendation)
print("Learner control: accept, review explanation, or choose another pathway.")


Estimated completion probability: 0.707
Recommendation: Advanced pathway recommended
Learner control: accept, review explanation, or choose another pathway.


## 9. Responsible interpretation

This prototype is intentionally limited.

It does **not** establish that:
- the chosen features validly measure learning;
- the model will generalise to real organisations;
- the probability estimates are calibrated;
- the same decision threshold is appropriate for every context;
- the model is fair across learner groups;
- a predicted risk tells us which intervention will improve outcomes.

These are part of the design problem, not afterthoughts.


## 10. Next step

A stronger next iteration would separate three questions:

1. **Prediction:** Who is likely to need support?
2. **Decision:** When should support be offered?
3. **Causal effect:** Which support actually improves learning for whom?

PathWise AI should only become more complex if additional complexity is justified by the learning problem and evidence.
